In [4]:
import qprogram as qp
from qprogram.buses import BusSchema
from qprogram.waveforms import IQDrag, IQPair, Square

# Setup bus schema for a flux-tunable transmon chip
schema = BusSchema.flux_tunable_transmon()
q = schema.q

# Create program using typed bus references and waveform aliases
program = qp.QProgram(label="rabi", description="Rabi oscillation")
gain = program.variable("gain")

with program.average(shots=1000):
    with program.for_loop(gain, start=0.0, stop=1.0, step=0.01):
        program.set_gain(q[0].drive, gain)
        program.play(q[0].drive, "pi_pulse")
        program.sync()
        program.measure(q[0].readout, "readout", "weights")

# Save to file
qp.save(program, "rabi.qp")

# Resolve waveform aliases with concrete values (e.g. from calibration data)
resolved = program.with_waveforms({
    "pi_pulse": IQDrag(0.5, 40, 2.5, 0.1),
    "readout": IQPair(Square(1.0, 2000), Square(0.0, 2000)),
    "weights": IQPair(Square(1.0, 2000), Square(1.0, 2000)),
})

qp.save(resolved, "rabi_resolved.qp")

In [5]:
program.schema

BusSchema(q(drive: ('IQ', False), readout: ('IQ', True), flux: ('single', False)))